# RO1 — Multi-Source Data Fusion for Stock Market Prediction

**Research objective (RO1):** *"To investigate and develop a comprehensive model that integrates diverse data
sources — financial metrics, news articles, social media sentiment, and macroeconomic factors — aiming to
enhance the accuracy of stock market predictions by providing a holistic analysis."*

**Base paper:** *A Multi-Source Data Fusion Framework for Enhanced Stock Market Prediction* (MSFN) —
CNN/technical + transformer-sentiment + macro streams fused via cross-modal attention into a BiLSTM head.

**What this notebook actually does (simplified, real-data version):**
- Fetches **real** NSE price history (15 liquid NIFTY50 names), **real** quarterly fundamentals and earnings
  surprises, and **real** India macro data (FRED: USD/INR, GDP growth) + a documented RBI repo-rate history —
  no simulated/synthetic rows anywhere.
- Predicts the **21-trading-day ("~30-day swing") forward return**, not next-day. This project's own
  production ledger found essentially no usable edge at 1-day horizons (next-period direction ≈ 50%, ranking
  AUC ≈ 0.47) and real, gate-able edge specifically at a ~30-day swing horizon — this notebook targets the
  same horizon so results are structurally comparable to that finding rather than a different, easier one.
- Fuses **technical + fundamental + macro** (the three sources with genuine multi-year coverage) with a small
  cross-modal-attention network (GRU technical encoder + MLP encoders + learned attention, instead of the
  paper's heavier CNN/BiLSTM stack — this keeps training under ~2 minutes on a free Colab T4).
- Appends a short, honestly-scoped **sentiment** experiment: free news APIs only return a recent window of
  headlines (weeks, not years) per ticker, so the 4th modality (news/social sentiment) is added and evaluated
  **only** on that recent window, side-by-side against the same 3-source model on the same dates — not baked
  into the long-history backbone where it would be silently imputed as neutral 95% of the time.
- Reports honest walk-forward metrics against naive baselines. **We do not expect to match the original
  paper's headline numbers** (it does not disclose data vendor/leakage details) — the goal is a correct,
  reproducible, presentation-ready pipeline on data anyone can re-pull for free, whose *findings* are
  consistent with this project's own already-validated production results.

Runtime: ~3-5 minutes end-to-end on a free Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!pip -q install yfinance==0.2.* --upgrade
import warnings; warnings.filterwarnings("ignore")
print("done")

In [ ]:
import numpy as np
import pandas as pd
import requests, io, time
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

TICKERS = ["RELIANCE.NS","TCS.NS","HDFCBANK.NS","INFY.NS","ICICIBANK.NS",
           "ITC.NS","LT.NS","SBIN.NS","BHARTIARTL.NS","HINDUNILVR.NS",
           "KOTAKBANK.NS","AXISBANK.NS","MARUTI.NS","SUNPHARMA.NS","TATASTEEL.NS"]
START = "2015-01-01"
HORIZON = 21     # ~30 calendar days -- the swing horizon where this project's production ledger found real edge
LOOKBACK = 30    # technical encoder window length
TRAIN_END = "2022-01-01"
VAL_END   = "2023-01-01"
# test = everything after VAL_END, up to whatever "today" is when you run this

### 1. Real price history + technical indicators (yfinance, NSE)

In [ ]:
import yfinance as yf

def fetch_prices(ticker, start=START):
    df = yf.download(ticker, start=start, progress=False, auto_adjust=True)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df = df.dropna(how="all")
    return df

def add_technical(df):
    out = df.copy()
    out["ret1"] = out["Close"].pct_change()
    out["sma20"] = out["Close"].rolling(20).mean() / out["Close"] - 1
    out["ema20"] = out["Close"].ewm(span=20).mean() / out["Close"] - 1
    delta = out["Close"].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = (-delta.clip(upper=0)).rolling(14).mean()
    rs = up / down.replace(0, np.nan)
    out["rsi14"] = (100 - (100 / (1 + rs))) / 100.0
    ema12 = out["Close"].ewm(span=12).mean()
    ema26 = out["Close"].ewm(span=26).mean()
    out["macd"] = (ema12 - ema26) / out["Close"]
    std20 = out["Close"].rolling(20).std()
    out["bb_pctb"] = (out["Close"] - (out["sma20"]*out["Close"]+out["Close"] - 2*std20)) / (4*std20 + 1e-9)
    out["vol20"] = out["ret1"].rolling(20).std() * np.sqrt(252)
    return out

TECH_COLS = ["ret1","sma20","ema20","rsi14","macd","bb_pctb","vol20"]

price_data = {}
for tk in TICKERS:
    df = add_technical(fetch_prices(tk))
    price_data[tk] = df
    print(tk, len(df), "rows", df.index.min().date(), "->", df.index.max().date())

### 2. Real fundamentals — quarterly financials + earnings surprise (yfinance)

In [ ]:
def fetch_fundamentals(ticker):
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials
    fdf = pd.DataFrame(index=pd.DatetimeIndex([]))
    if qf is not None and not qf.empty:
        rows = {}
        for key in ["Total Revenue", "Net Income"]:
            if key in qf.index:
                rows[key] = qf.loc[key].sort_index()
        if rows:
            fdf = pd.DataFrame(rows).sort_index()
            fdf["rev_growth_qoq"] = fdf.get("Total Revenue", pd.Series(dtype=float)).pct_change()
            fdf["ni_growth_qoq"] = fdf.get("Net Income", pd.Series(dtype=float)).pct_change()
    try:
        ed = t.get_earnings_dates(limit=40)[["Surprise(%)"]].dropna().sort_index()
        ed.index = ed.index.tz_localize(None)
        ed.columns = ["eps_surprise_pct"]
    except Exception:
        ed = pd.DataFrame(columns=["eps_surprise_pct"])
    return fdf[["rev_growth_qoq","ni_growth_qoq"]] if not fdf.empty else fdf, ed

FUND_COLS = ["rev_growth_qoq","ni_growth_qoq","eps_surprise_pct"]
fund_data = {}
for tk in TICKERS:
    fdf, ed = fetch_fundamentals(tk)
    daily_idx = price_data[tk].index
    rev = fdf["rev_growth_qoq"].reindex(daily_idx, method="ffill") if "rev_growth_qoq" in fdf else pd.Series(np.nan, index=daily_idx)
    ni  = fdf["ni_growth_qoq"].reindex(daily_idx, method="ffill") if "ni_growth_qoq" in fdf else pd.Series(np.nan, index=daily_idx)
    eps = ed["eps_surprise_pct"].reindex(daily_idx, method="ffill") if not ed.empty else pd.Series(np.nan, index=daily_idx)
    fund_data[tk] = pd.DataFrame({"rev_growth_qoq": rev, "ni_growth_qoq": ni, "eps_surprise_pct": eps})
    cov = fund_data[tk].notna().mean().mean()
    print(tk, f"fundamental coverage: {cov:.0%}")

### 3. Real macro data — USD/INR, India GDP growth (FRED, no API key) + RBI repo rate

`INTDSRINM193N` (India discount rate) stops updating in 2022 in FRED's series, so it is not usable as a live
policy-rate feed for recent years. Instead we use FRED's daily USD/INR rate and quarterly GDP growth (both
verified live below), plus a **manually documented table of actual RBI Monetary Policy Committee repo-rate
decisions** (public record, rbi.org.in). Rate values are accurate; day-of-month precision for pre-2019 entries
may be off by a few days depending on source — negligible at daily-model resolution, but re-verify against
rbi.org.in if you need exact-day precision. **Nothing here is invented** — it is either pulled live or a
documented historical fact.

In [ ]:
def fetch_fred(series_id):
    r = requests.get(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}", timeout=20)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df.set_index("date")[series_id].dropna()

usdinr = fetch_fred("DEXINUS")
usdinr_chg5 = usdinr.pct_change(5)
gdp_growth = fetch_fred("INDGDPRQPSMEI")
gdp_growth.index = gdp_growth.index + pd.Timedelta(days=60)   # simulate real publication lag, avoid lookahead

RBI_REPO_RATE = pd.DataFrame({
    "date": ["2014-01-01","2015-01-15","2015-03-04","2015-06-02","2015-09-29","2016-04-05",
             "2017-08-02","2018-06-06","2018-08-01","2019-02-07","2019-04-04","2019-06-06",
             "2019-08-07","2019-10-04","2020-03-27","2020-05-22","2022-05-04","2022-06-08",
             "2022-08-05","2022-09-30","2022-12-07","2023-02-08"],
    "repo_rate": [8.00,7.75,7.50,7.25,6.75,6.50,6.00,6.25,6.50,6.25,6.00,5.75,
                  5.40,5.15,4.40,4.00,4.40,4.90,5.40,5.90,6.25,6.50],
})
RBI_REPO_RATE["date"] = pd.to_datetime(RBI_REPO_RATE["date"])
repo_series = RBI_REPO_RATE.set_index("date")["repo_rate"].sort_index()

print("USD/INR through", usdinr.index.max().date(), "| GDP growth through", (gdp_growth.index.max()-pd.Timedelta(days=60)).date(),
      "| repo rate through", repo_series.index.max().date(), f"({repo_series.iloc[-1]}%, verify vs rbi.org.in if run much later)")

MACRO_COLS = ["usdinr_chg5","gdp_growth","repo_rate"]
macro_data = {}
for tk in TICKERS:
    idx = price_data[tk].index
    macro_data[tk] = pd.DataFrame({
        "usdinr_chg5": usdinr_chg5.reindex(idx, method="ffill"),
        "gdp_growth": gdp_growth.reindex(idx, method="ffill"),
        "repo_rate": repo_series.reindex(idx, method="ffill"),
    })

### 4. Assemble the panel + forward-return label

In [ ]:
panels = {}
for tk in TICKERS:
    df = price_data[tk][["Close"] + TECH_COLS].join(fund_data[tk]).join(macro_data[tk])
    df["fwd_ret"] = df["Close"].shift(-HORIZON) / df["Close"] - 1
    df["ticker"] = tk
    panels[tk] = df

full = pd.concat(panels.values()).sort_index()
FEATURE_COLS = TECH_COLS + FUND_COLS + MACRO_COLS
# fundamentals can be legitimately missing before a company's first covered quarter -> fill with 0 (neutral), not dropped
full[FUND_COLS] = full[FUND_COLS].fillna(0.0)
full = full.dropna(subset=TECH_COLS + MACRO_COLS + ["fwd_ret"])
print(full.shape, "rows across", full['ticker'].nunique(), "tickers")
full.tail(3)

### 5. Windowed dataset, walk-forward split (by date, no shuffling across time)

In [ ]:
def make_windows(panel_df, lookback=LOOKBACK):
    X_tech, X_fund, X_macro, y, dates, tickers = [], [], [], [], [], []
    for tk, g in panel_df.groupby("ticker"):
        g = g.sort_index()
        tech_arr = g[TECH_COLS].values
        fund_arr = g[FUND_COLS].values
        macro_arr = g[MACRO_COLS].values
        y_arr = g["fwd_ret"].values
        idx = g.index
        for i in range(lookback, len(g)):
            X_tech.append(tech_arr[i-lookback:i])
            X_fund.append(fund_arr[i])
            X_macro.append(macro_arr[i])
            y.append(y_arr[i])
            dates.append(idx[i])
            tickers.append(tk)
    return (np.array(X_tech, dtype=np.float32), np.array(X_fund, dtype=np.float32),
            np.array(X_macro, dtype=np.float32), np.array(y, dtype=np.float32),
            pd.DatetimeIndex(dates), np.array(tickers))

X_tech, X_fund, X_macro, y, dates, tk_arr = make_windows(full)
print("windows:", X_tech.shape, X_fund.shape, X_macro.shape, y.shape)

train_mask = dates < TRAIN_END
val_mask   = (dates >= TRAIN_END) & (dates < VAL_END)
test_mask  = dates >= VAL_END
print("train/val/test:", train_mask.sum(), val_mask.sum(), test_mask.sum())

# standardize technical/fundamental/macro features using TRAIN stats only (no leakage)
tech_scaler = StandardScaler().fit(X_tech[train_mask].reshape(-1, X_tech.shape[-1]))
fund_scaler = StandardScaler().fit(X_fund[train_mask])
macro_scaler = StandardScaler().fit(X_macro[train_mask])

def scale_tech(a):
    shp = a.shape
    return tech_scaler.transform(a.reshape(-1, shp[-1])).reshape(shp).astype(np.float32)

X_tech_s = scale_tech(X_tech)
X_fund_s = fund_scaler.transform(X_fund).astype(np.float32)
X_macro_s = macro_scaler.transform(X_macro).astype(np.float32)

In [ ]:
class PanelDataset(Dataset):
    def __init__(self, mask):
        self.xt = X_tech_s[mask]; self.xf = X_fund_s[mask]; self.xm = X_macro_s[mask]; self.y = y[mask]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.xt[i], self.xf[i], self.xm[i], self.y[i]

train_ds, val_ds, test_ds = PanelDataset(train_mask), PanelDataset(val_mask), PanelDataset(test_mask)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=512)
test_dl  = DataLoader(test_ds, batch_size=512)
print(len(train_ds), len(val_ds), len(test_ds))

### 6. Cross-modal attention fusion network (technical GRU + fundamental/macro MLPs + attention)

In [ ]:
class TechEncoder(nn.Module):
    def __init__(self, n_feat, hidden=32):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, batch_first=True)
    def forward(self, x):
        _, h = self.gru(x)
        return h.squeeze(0)

class MLPEncoder(nn.Module):
    def __init__(self, n_feat, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_feat, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
    def forward(self, x):
        return self.net(x)

class CrossModalAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.score = nn.Linear(hidden, 1)
    def forward(self, modality_embeds):
        stacked = torch.stack(modality_embeds, dim=1)      # (B, M, H)
        scores = self.score(stacked).squeeze(-1)           # (B, M)
        weights = torch.softmax(scores, dim=1)              # (B, M)
        fused = (stacked * weights.unsqueeze(-1)).sum(dim=1)
        return fused, weights

class MSFN(nn.Module):
    def __init__(self, n_tech, n_fund, n_macro, hidden=32, extra_encoders=None):
        super().__init__()
        self.tech_enc = TechEncoder(n_tech, hidden)
        self.fund_enc = MLPEncoder(n_fund, hidden)
        self.macro_enc = MLPEncoder(n_macro, hidden)
        self.extra_encoders = nn.ModuleList(extra_encoders) if extra_encoders else nn.ModuleList([])
        self.attn = CrossModalAttention(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x_tech, x_fund, x_macro, extra_inputs=None):
        embeds = [self.tech_enc(x_tech), self.fund_enc(x_fund), self.macro_enc(x_macro)]
        if extra_inputs is not None:
            for enc, xin in zip(self.extra_encoders, extra_inputs):
                embeds.append(enc(xin))
        fused, weights = self.attn(embeds)
        return self.head(fused).squeeze(-1), weights

model = MSFN(len(TECH_COLS), len(FUND_COLS), len(MACRO_COLS)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.SmoothL1Loss()
print(model)

In [ ]:
def run_epoch(dl, train=True):
    model.train(train)
    total, n = 0.0, 0
    for xt, xf, xm, yb in dl:
        xt, xf, xm, yb = xt.to(DEVICE), xf.to(DEVICE), xm.to(DEVICE), yb.to(DEVICE)
        if train: opt.zero_grad()
        pred, _ = model(xt, xf, xm)
        loss = loss_fn(pred, yb)
        if train:
            loss.backward(); opt.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n

best_val, patience, bad_epochs = float("inf"), 5, 0
history = []
for epoch in range(30):
    tr_loss = run_epoch(train_dl, train=True)
    val_loss = run_epoch(val_dl, train=False)
    history.append((tr_loss, val_loss))
    if val_loss < best_val - 1e-6:
        best_val, bad_epochs = val_loss, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
    print(f"epoch {epoch:02d}  train {tr_loss:.5f}  val {val_loss:.5f}")
    if bad_epochs >= patience:
        print("early stopping"); break
model.load_state_dict(best_state)

### 7. Honest evaluation on the untouched test window (vs naive baselines)

In [ ]:
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for xt, xf, xm, yb in test_dl:
        p, _ = model(xt.to(DEVICE), xf.to(DEVICE), xm.to(DEVICE))
        preds.append(p.cpu().numpy()); actuals.append(yb.numpy())
preds = np.concatenate(preds); actuals = np.concatenate(actuals)

train_mean = y[train_mask].mean()
baseline_zero = np.zeros_like(actuals)
baseline_mean = np.full_like(actuals, train_mean)

def report(name, p, a):
    mae = mean_absolute_error(a, p); rmse = mean_squared_error(a, p) ** 0.5
    r2 = r2_score(a, p)
    dir_acc = (np.sign(p) == np.sign(a)).mean()
    print(f"{name:22s}  MAE {mae:.5f}  RMSE {rmse:.5f}  R2 {r2:+.4f}  dir_acc {dir_acc:.3%}")

print(f"test window: {dates[test_mask].min().date()} -> {dates[test_mask].max().date()}  (n={test_mask.sum()})")
print(f"base rate (actual up-moves): {(actuals > 0).mean():.3%}")
report("MSFN (3-source fusion)", preds, actuals)
report("baseline: zero return", baseline_zero, actuals)
report("baseline: train mean", baseline_mean, actuals)

plt.figure(figsize=(9,4))
plt.scatter(actuals, preds, s=4, alpha=0.3)
lims = [actuals.min(), actuals.max()]
plt.plot(lims, lims, "r--", lw=1)
plt.xlabel(f"actual {HORIZON}-day fwd return"); plt.ylabel("predicted"); plt.title("MSFN: predicted vs actual (test set)")
plt.tight_layout(); plt.show()

**Read this honestly, not optimistically.** If directional accuracy on the test window sits close to the
base up-move rate, the model is not finding edge beyond what a naive baseline already captures — that is a
real, common finding for single-name daily/weekly direction (consistent with prior work on this exact data:
next-period direction is close to a coin flip once drift is accounted for). Report R² and MAE alongside
directional accuracy rather than directional accuracy alone, and do not present a near-baseline number as a
breakthrough in a presentation.

### 8. Feature-occlusion ablation — how much does each source actually contribute?

We zero out one modality's *input* at inference time (on the already-trained model) and re-measure test MAE.
This is a fast proxy ablation (occlusion), not a full leave-one-source-out retrain — say so explicitly if you present it.

In [ ]:
def evaluate_occluded(zero_tech=False, zero_fund=False, zero_macro=False):
    preds = []
    with torch.no_grad():
        for xt, xf, xm, yb in test_dl:
            xt, xf, xm = xt.clone(), xf.clone(), xm.clone()
            if zero_tech: xt.zero_()
            if zero_fund: xf.zero_()
            if zero_macro: xm.zero_()
            p, _ = model(xt.to(DEVICE), xf.to(DEVICE), xm.to(DEVICE))
            preds.append(p.cpu().numpy())
    preds = np.concatenate(preds)
    return mean_absolute_error(actuals, preds)

full_mae = mean_absolute_error(actuals, preds)
rows = [("all 3 sources", full_mae),
        ("no technical", evaluate_occluded(zero_tech=True)),
        ("no fundamental", evaluate_occluded(zero_fund=True)),
        ("no macro", evaluate_occluded(zero_macro=True))]
for name, mae in rows:
    print(f"{name:16s} test MAE {mae:.5f}  (worse than full model: {mae > full_mae})")

# attention weights actually learned, averaged over the test set
w_list = []
with torch.no_grad():
    for xt, xf, xm, yb in test_dl:
        _, w = model(xt.to(DEVICE), xf.to(DEVICE), xm.to(DEVICE))
        w_list.append(w.cpu().numpy())
w_all = np.concatenate(w_list)
plt.figure(figsize=(5,4))
plt.bar(["technical","fundamental","macro"], w_all.mean(axis=0))
plt.ylabel("mean learned attention weight"); plt.title("Cross-modal attention weights (test set avg)")
plt.tight_layout(); plt.show()

## 9. Sentiment as a 4th source — honestly scoped to the window where real news actually exists

Free news sources (Yahoo Finance's `Ticker.news`) only return a **recent window** of real headlines per
ticker (typically the trailing few weeks), not multi-year archives. Rather than back-filling ~10 years of
"neutral" sentiment (which would be misleading — it would look like data, but it would just be an artifact
of missing coverage), we run a **separate, smaller, honestly-labelled experiment**: refit a lightweight
version of the same fusion architecture with sentiment added as a 4th modality, evaluated only on the
window where real headlines exist, against the same 3-source model restricted to those same dates.

In [ ]:
!pip -q install transformers==4.* accelerate --upgrade
from transformers import pipeline
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
POLARITY_SIGN = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}

MATERIALITY_KEYWORDS = ["earnings","profit","loss","guidance","acquisition","merger","stake","ipo",
                         "dividend","buyback","lawsuit","regulatory","rbi","repo","downgrade","upgrade",
                         "rating","fraud","default","ceo","resign","results","revenue"]

def materiality_score(title):
    t = title.lower()
    hits = sum(1 for kw in MATERIALITY_KEYWORDS if kw in t)
    return min(1.0, hits / 3.0)

def fetch_news_sentiment(ticker):
    try:
        news = yf.Ticker(ticker).news
    except Exception:
        news = []
    rows = []
    for n in news:
        c = n.get("content", {})
        title = c.get("title"); pub = c.get("pubDate")
        if not title or not pub:
            continue
        rows.append({"date": pd.to_datetime(pub).tz_localize(None), "title": title})
    if not rows:
        return pd.DataFrame(columns=["date","title","polarity","materiality"])
    df = pd.DataFrame(rows).sort_values("date")
    sentiments = finbert(df["title"].tolist())
    df["polarity"] = [POLARITY_SIGN[s["label"]] * s["score"] for s in sentiments]
    df["materiality"] = df["title"].apply(materiality_score)
    return df

news_frames = {tk: fetch_news_sentiment(tk) for tk in TICKERS}
total_headlines = sum(len(v) for v in news_frames.values())
print(f"real headlines fetched across {len(TICKERS)} tickers: {total_headlines}")
for tk, v in news_frames.items():
    if len(v):
        print(f"  {tk}: {len(v)} headlines, {v['date'].min().date()} -> {v['date'].max().date()}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def add_novelty(df, window=5):
    if len(df) < 2:
        df["novelty"] = 1.0
        return df
    vec = TfidfVectorizer(stop_words="english")
    tfidf = vec.fit_transform(df["title"])
    novelty = []
    for i in range(len(df)):
        lo = max(0, i - window)
        if i == lo:
            novelty.append(1.0); continue
        sims = cosine_similarity(tfidf[i], tfidf[lo:i])
        novelty.append(float(1.0 - sims.max()))
    df["novelty"] = novelty
    return df

for tk in TICKERS:
    news_frames[tk] = add_novelty(news_frames[tk])

# daily gate = polarity * novelty * materiality (paper's multiplicative-gating idea), aggregated to 1/day
sent_daily = {}
for tk in TICKERS:
    df = news_frames[tk]
    if df.empty:
        sent_daily[tk] = pd.DataFrame(columns=["polarity","novelty","materiality","gate","has_news"])
        continue
    df["gate"] = df["polarity"] * df["novelty"] * df["materiality"]
    daily = df.groupby(df["date"].dt.floor("D"))[["polarity","novelty","materiality","gate"]].mean()
    daily["has_news"] = 1.0
    sent_daily[tk] = daily

SENT_COLS = ["polarity","novelty","materiality","gate","has_news"]

In [ ]:
# recent-window comparison: same dates, with vs without the sentiment modality
recent_frames = []
for tk in TICKERS:
    idx = price_data[tk].index
    sd = sent_daily[tk].reindex(idx, method=None)  # no ffill: absence of news that day is real information, not missingness
    sd = sd.fillna({"polarity":0.0,"novelty":0.0,"materiality":0.0,"gate":0.0,"has_news":0.0})
    covered_dates = sd.index[sd["has_news"] == 1.0]
    recent_frames.append(sd.loc[covered_dates].assign(ticker=tk))

sent_panel = pd.concat(recent_frames).sort_index() if any(len(f) for f in recent_frames) else pd.DataFrame()
print("sentiment-covered rows across all tickers:", len(sent_panel))
if len(sent_panel) < 200:
    print("NOTE: coverage is thin (free news API + few tickers). Treat the numbers below as illustrative,"
          " not a statistically powerful test -- this is a real limitation of free data sources, not hidden.")

In [ ]:
if len(sent_panel) >= 40:
    # Attach sentiment features to the ALREADY correctly-windowed technical/fundamental/macro
    # arrays from cell-12 (30 contiguous trading days pulled from the FULL continuous price
    # history), instead of rebuilding windows from the sparse news-only panel. The news panel
    # only has a handful of rows per ticker (weeks apart, not consecutive trading days), so a
    # 30-day lookback window can never be satisfied there -- that's why make_windows(sent_full)
    # produced an empty array (0 windows for every ticker) and the reshape crashed.
    sent_lookup = {}
    for tk in TICKERS:
        sd = sent_daily[tk]
        if sd.empty:
            continue
        for dt, row in sd.iterrows():
            sent_lookup[(tk, dt)] = row[SENT_COLS].values.astype(np.float32)

    keep_mask = np.array([(tk_arr[i], dates[i]) in sent_lookup for i in range(len(dates))])
    print("recent-window fused rows (technical window intact, sentiment attached by date):", keep_mask.sum())

    if keep_mask.sum() >= 40:
        X_tech_r = X_tech[keep_mask]; X_fund_r = X_fund[keep_mask]; X_macro_r = X_macro[keep_mask]
        y_r = y[keep_mask]; dates_r = dates[keep_mask]; tk_r = tk_arr[keep_mask]
        X_sent_r = np.stack([sent_lookup[(tk_r[i], dates_r[i])] for i in range(keep_mask.sum())])
        print("aligned shapes:", X_tech_r.shape, X_sent_r.shape)

        n = len(y_r)
        cut = int(n * 0.7)  # small sample -> simple time-ordered 70/30 split instead of the 3-way split above
        tr, te = slice(0, cut), slice(cut, n)

        def small_eval(use_sentiment):
            sc_t = StandardScaler().fit(X_tech_r[tr].reshape(-1, X_tech_r.shape[-1]))
            sc_f = StandardScaler().fit(X_fund_r[tr]); sc_m = StandardScaler().fit(X_macro_r[tr])
            xt = sc_t.transform(X_tech_r.reshape(-1, X_tech_r.shape[-1])).reshape(X_tech_r.shape).astype(np.float32)
            xf = sc_f.transform(X_fund_r).astype(np.float32); xm = sc_m.transform(X_macro_r).astype(np.float32)
            extra_encoders = []
            if use_sentiment:
                sc_s = StandardScaler().fit(X_sent_r[tr])
                xs = sc_s.transform(X_sent_r).astype(np.float32)
                extra_encoders = [MLPEncoder(X_sent_r.shape[-1], 32)]
            m = MSFN(len(TECH_COLS), len(FUND_COLS), len(MACRO_COLS), extra_encoders=extra_encoders).to(DEVICE)
            o = torch.optim.Adam(m.parameters(), lr=1e-3)
            xt_t, xf_t, xm_t, y_t = [torch.tensor(a[tr]) for a in (xt, xf, xm, y_r)]
            xt_e, xf_e, xm_e, y_e = [torch.tensor(a[te]) for a in (xt, xf, xm, y_r)]
            xs_t = torch.tensor(xs[tr]) if use_sentiment else None
            xs_e = torch.tensor(xs[te]) if use_sentiment else None
            for epoch in range(20):
                m.train()
                o.zero_grad()
                extra = [xs_t.to(DEVICE)] if use_sentiment else None
                p, _ = m(xt_t.to(DEVICE), xf_t.to(DEVICE), xm_t.to(DEVICE), extra)
                l = nn.SmoothL1Loss()(p, y_t.to(DEVICE))
                l.backward(); o.step()
            m.eval()
            with torch.no_grad():
                extra = [xs_e.to(DEVICE)] if use_sentiment else None
                p, _ = m(xt_e.to(DEVICE), xf_e.to(DEVICE), xm_e.to(DEVICE), extra)
                p = p.cpu().numpy()
            mae = mean_absolute_error(y_r[te], p)
            dir_acc = (np.sign(p) == np.sign(y_r[te])).mean()
            return mae, dir_acc

        mae_without, dir_without = small_eval(use_sentiment=False)
        mae_with, dir_with = small_eval(use_sentiment=True)
        print(f"n_test={n-cut} (recent, news-covered dates only)")
        print(f"3 sources (no sentiment):  MAE {mae_without:.5f}  dir_acc {dir_without:.3%}")
        print(f"4 sources (+ sentiment):   MAE {mae_with:.5f}  dir_acc {dir_with:.3%}")
        print("Report whichever is actually better -- do not round in favor of the sentiment story.")
    else:
        print("No date+ticker pair with real news lined up with a full 30-day technical lookback -- skipping the sentiment comparison.")
else:
    print("Sentiment coverage too thin this run to fit a second model honestly -- reporting headline count only.")

## Summary for a presentation slide

- **Data**: 100% real — NSE prices via yfinance, quarterly fundamentals + EPS surprises via yfinance,
  India macro (USD/INR, GDP growth) via FRED, RBI repo-rate from documented MPC decisions, news headlines via
  Yahoo Finance + FinBERT/TF-IDF-derived sentiment. No row was simulated.
- **Model**: cross-modal attention fusion (GRU-technical + MLP-fundamental + MLP-macro [+ MLP-sentiment]),
  a simplified stand-in for the paper's CNN/Transformer/BiLSTM stack, chosen so the whole notebook trains
  in minutes on a free T4.
- **Headline numbers to quote**: pull the printed MAE / RMSE / R² / directional-accuracy lines above (Section 7)
  and the ablation table (Section 8) directly — do not restate the original paper's claimed numbers as if this
  notebook reproduced them.
- **Known limitation to disclose**: sentiment coverage is limited to whatever the free API returns at run time
  (typically a few recent weeks per ticker); the sentiment comparison in Section 9 is illustrative, not a
  large-sample result.